In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(df.shape)
df.head(15)

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,0
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.90,0
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,0


In [3]:
# 1. Сегментация по сроку
def tenure_segment(t):
    if t <= 12: return 'new'
    elif t <= 36: return 'middle'
    else: return 'loyal'

df['tenure_segment'] = df['tenure'].apply(tenure_segment)

# 2. Цена за сервис
df['monthly_charges_per_service'] = (
    df['MonthlyCharges'] /
    (df[['PhoneService', 'MultipleLines', 'InternetService', 
         'TechSupport', 'StreamingTV', 'StreamingMovies']]
    == 'Yes').sum(axis=1).replace(0, 1)
)

# 3. Флаг множества контрактов
df['has_multiple_contracts'] = (
    (df['PhoneService'] == 'Yes') &
    (df['InternetService'] != 'No')
).astype(int)

print(df[['tenure_segment', 'monthly_charges_per_service',
          'has_multiple_contracts']].head(15))

   tenure_segment  monthly_charges_per_service  has_multiple_contracts
0             new                      29.8500                       0
1          middle                      56.9500                       1
2             new                      53.8500                       1
3           loyal                      42.3000                       0
4             new                      70.7000                       1
5             new                      24.9125                       1
6          middle                      29.7000                       1
7             new                      29.7500                       0
8          middle                      20.9600                       1
9           loyal                      56.1500                       1
10         middle                      49.9500                       1
11         middle                      18.9500                       0
12          loyal                      25.0875                       1
13    

In [4]:
df.to_csv('../data/processed/telco_featured.csv', index=False)
print(f"Сохранено: {df.shape}")

Сохранено: (7043, 24)
